# 9장 실습: Vector + Graph 하이브리드 RAG

실습 가이드 9장 9-2 ~ 9-4의 코드를 순서대로 실행하는 노트북입니다. 설명은 실습 가이드를 함께 봅니다.

| 항목 | 내용 |
|---|---|
| 연결할 DB | Recommendations (APOC 포함, 쓰기 권한 필요) |
| .env 설정 | Recommendations DB의 NEO4J 네 줄 (로컬 Docker라면 `bolt://localhost:7687`) |

.env를 바꿨다면 노트북 상단의 `Restart`로 커널을 재시작한 뒤 처음부터 실행합니다. 연결 셀의 `connect_graph()`는 .env를 다시 읽지만, 앞 셀에서 만든 객체가 이전 DB를 가리킬 수 있으므로 처음부터 다시 실행합니다.

줄거리 임베딩이 아직 없는 DB라면 아래 1번 셀에서 한 번 만듭니다. 이미 있으면 그대로 넘어갑니다.

## 연결 확인

`Neo4jGraph`는 연결할 때 DB 스키마를 읽습니다. 이때 APOC 플러그인을 쓰므로, APOC가 없는 DB에서는 `Could not use APOC procedures` 오류가 납니다(Sandbox, AuraDB, 실습용 Docker 이미지에는 들어 있습니다).

출력된 레이블별 노드 수로 이 차시에 맞는 DB에 연결됐는지 확인합니다.

In [ ]:
from tools.graphrag_tools import connect_graph

# .env의 접속 정보로 연결하고 레이블별 노드 수를 출력한다. expect의 레이블이 없으면 다른 DB에 연결됐다고 알려 준다.
# NEO4J는 접속 정보 묶음이다. 다른 LangChain 객체를 만들 때 **NEO4J로 그대로 넘긴다(실습 가이드 6-2).
graph, NEO4J = connect_graph(expect=["Movie", "User", "Genre"])

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_neo4j import Neo4jVector, GraphCypherQAChain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda

llm = ChatOpenAI(model="gpt-4o", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## 1. 줄거리 임베딩 (실습 가이드 9-2)

9장의 벡터 검색은 영화 줄거리를 벡터로 바꿔 둔 `Movie.embedding` 속성과 벡터 인덱스 `moviePlotIndex`가 있어야 동작합니다. 아래 셀의 `Neo4jVector.from_existing_graph`가 이 두 가지를 만듭니다. 내부에서는 다음 순서로 처리합니다.

1. `moviePlotIndex`가 없으면 만듭니다(1,536차원, cosine).
2. `embedding`이 없는 Movie를 1,000편씩 가져옵니다. 임베딩할 텍스트는 영화마다 `"\nplot:<줄거리>"` 형식입니다.
3. 1,000편의 텍스트를 OpenAI 임베딩 API에 한 번에 보내 벡터 1,000개를 받습니다.
4. `db.create.setNodeVectorProperty`로 각 Movie의 `embedding` 속성에 저장합니다.
5. 남은 영화가 없을 때까지 2~4를 반복합니다. 9,083편이면 10번입니다.

이미 임베딩된 영화는 건너뛰므로, 다시 실행해도 비용이 다시 들지 않고 중간에 끊겨도 이어서 할 수 있습니다.

| 항목 | 값 (2026-11-01 측정) |
|---|---|
| 임베딩 대상 | plot이 있는 영화 9,083편 (전체 9,125편) |
| 임베딩 텍스트 | 9,083개, 합계 315,376토큰 (편당 평균 35토큰) |
| 비용 | 1명 약 $0.006 (text-embedding-3-small, 100만 토큰당 $0.02 기준) |
| 시간 | 로컬 Docker에서 이 노트북 전체 실행 83초 |

학생이 직접 실행해도 되는가

비용과 시간은 문제가 되지 않습니다. 조건은 학생마다 쓰기 권한이 있는 Recommendations DB(APOC 포함)가 하나씩 있어야 한다는 것입니다.

| 운영 방식 | 방법 | 비고 |
|---|---|---|
| 학생이 직접 실행 | 아래 `RUN_EMBEDDING = True`로 한 번 실행 | 각자 DB가 필요합니다. Docker는 덤프 적재 1초, AuraDB는 Push to Aura에 수십 분이 걸립니다 |
| 임베딩된 DB를 나눠 줌 | 강사가 임베딩까지 마친 DB를 덤프 파일로 만들어 배포 | 학생은 Docker에 적재하거나 AuraDB 콘솔에서 가져오기만 하면 됩니다 |
| 강사 DB 하나를 같이 씀 | 강사 DB 접속 정보를 공유 | AuraDB Free는 계정이 하나라 학생도 쓰기 쿼리를 실행할 수 있어 권하지 않습니다 |

아래 셀은 먼저 Movie 노드가 있는지 확인합니다. 없으면 Recommendations가 아닌 DB에 연결된 것이므로 멈춥니다.

In [ ]:
movie_count = graph.query("MATCH (m:Movie) RETURN count(m) AS n")[0]["n"]
if movie_count == 0:
    raise RuntimeError("Movie 노드가 없습니다. .env가 Recommendations DB를 가리키는지 확인하고 커널을 재시작하세요.")

# True로 바꾸면 임베딩이 없는 영화만 임베딩한다 (처음 1회, 약 $0.006, 1~2분)
RUN_EMBEDDING = False

if RUN_EMBEDDING:
    Neo4jVector.from_existing_graph(
        embedding=embeddings,
        **NEO4J,
        index_name="moviePlotIndex",
        node_label="Movie",
        text_node_properties=["plot"],
        embedding_node_property="embedding",
    )

embedded = graph.query("MATCH (m:Movie) WHERE m.embedding IS NOT NULL RETURN count(m) AS n")[0]["n"]
print(f"Movie {movie_count}편 중 임베딩된 영화 {embedded}편")
if embedded == 0:
    print("임베딩이 없습니다. RUN_EMBEDDING = True로 바꿔 이 셀을 다시 실행하세요.")

## 2. retrieval_query로 관계 정보 확장 (실습 가이드 9-3)

`retrieval_query`는 벡터 검색 결과인 `node`와 `score`를 받아 이어서 실행되는 Cypher입니다. `text`, `score`, `metadata` 세 항목을 반드시 반환해야 하고, 중간 `WITH`에서 `score`를 넘기지 않으면 오류가 납니다.

In [ ]:
retrieval_query = """
WITH node, score
OPTIONAL MATCH (node)<-[:ACTED_IN]-(actor:Actor)
OPTIONAL MATCH (node)-[:IN_GENRE]->(genre:Genre)
WITH node, score, collect(DISTINCT actor.name) AS actors, collect(DISTINCT genre.name) AS genres
RETURN node.plot AS text,
       score,
       {title: node.title, actors: actors, genres: genres} AS metadata
"""

hybrid_store = Neo4jVector.from_existing_index(
    embedding=embeddings,
    index_name="moviePlotIndex",
    **NEO4J,
    retrieval_query=retrieval_query,
)
hybrid_retriever = hybrid_store.as_retriever(search_kwargs={"k": 5})

for d in hybrid_retriever.invoke("What is the movie about the magic jungle board game?"):
    print(d.metadata["title"], d.metadata["genres"], d.metadata["actors"][:3])

## 3. 벡터 문맥과 그래프 문맥을 합치는 LCEL 체인 (실습 가이드 9-4)

`RunnableParallel`이 벡터 검색과 Text2Cypher 그래프 조회를 함께 실행하고, 두 결과를 하나의 프롬프트에 넣습니다. 그래프 조회가 실패해도 벡터 문맥만으로 답하도록 예외를 처리합니다.

In [ ]:
movie_cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    validate_cypher=True,
    allow_dangerous_requests=True,
)

def format_movies(docs):
    return "\n\n".join(
        f"제목: {d.metadata['title']}\n장르: {', '.join(d.metadata['genres'])}\n"
        f"배우: {', '.join(d.metadata['actors'][:5])}\n줄거리: {d.page_content}"
        for d in docs
    )

def graph_context(question: str) -> str:
    try:
        return movie_cypher_chain.invoke({"query": question})["result"]
    except Exception as e:
        return f"그래프 조회 실패: {e}"

fusion_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "너는 영화 지식 도우미다. 벡터 검색 문맥과 그래프 조회 결과를 함께 고려해 사실에 근거한 한국어 답변을 작성한다. "
     "두 정보가 충돌하면 그래프 결과를 우선한다. 문맥에 없는 내용은 추측하지 않는다."),
    ("human",
     "질문: {question}\n\n[벡터 검색 문맥]\n{vector_context}\n\n[그래프 결과]\n{graph_context}\n\n"
     "요구사항: 가장 관련 있는 영화 1~3편의 제목, 장르, 주요 배우, 줄거리 요약을 제시하고 끝에 근거를 표기한다."),
])

hybrid_chain = (
    RunnableParallel(
        question=RunnablePassthrough(),
        vector_context=hybrid_retriever | format_movies,
        graph_context=RunnableLambda(graph_context),
    )
    | fusion_prompt
    | llm
    | StrOutputParser()
)

print(hybrid_chain.invoke("A cowboy doll and new spaceman figure in a boy's room?"))
print("=" * 40)
print(hybrid_chain.invoke("What is the movie about the board game?"))

## 실습 과제

위의 1~3번 셀을 모두 실행한 뒤 진행합니다. 과제마다 문제 아래의 빈 셀에 코드를 작성합니다.

| 과제 | 다루는 것 |
|---|---|
| 1 | `retrieval_query`에 관계 하나를 더 붙여 metadata 늘리기 |
| 2 | 벡터 검색 결과에서 4장의 협업 필터링(멀티홉)으로 확장하기 |
| 3 | 데이터에 없는 질문을 했을 때 하이브리드 RAG가 어떻게 답하는지 확인하고 고치기 |

### 과제 1. 감독 정보 추가

2번 셀의 `retrieval_query`를 고쳐, 검색된 영화의 감독 목록을 metadata의 `directors`로 함께 돌려주세요. 감독은 `(d:Director)-[:DIRECTED]->(node)` 관계로 연결되어 있습니다. "What is the movie about the magic jungle board game?"으로 검색해 영화 제목과 감독을 출력합니다.

In [ ]:
# 과제 1: 여기에 작성합니다


### 과제 2. 함께 좋아한 영화 추가 (협업 필터링)

4장 추천 쿼리는 "이 영화를 평가한 사용자가 평가한 다른 영화"를 찾는 2-hop 쿼리였습니다. 이를 `retrieval_query`에 넣어, 벡터 검색으로 찾은 영화마다 다음 조건의 영화 5편을 metadata의 `alsoLiked`로 돌려주세요.

- 검색된 영화에 별점 4.0 이상을 준 사용자가
- 별점 4.0 이상을 준 다른 영화
- 그런 사용자가 많은 순서로 5편

"What is the movie about the magic jungle board game?"으로 상위 3편을 검색해 제목과 `alsoLiked`를 출력합니다.

In [ ]:
# 과제 2: 여기에 작성합니다


### 과제 3. 데이터에 없는 영화를 물었을 때

3번 셀의 `fusion_prompt`에는 "문맥에 없는 내용은 추측하지 않는다"는 규칙이 있습니다. 이 규칙이 지켜지는지 확인합니다.

1. 데이터에 없을 법한 영화를 질문합니다. 예: "A cat becomes a space pirate and fights robot dolphins on Mars."
2. `hybrid_store.similarity_search_with_score()`로 벡터 검색 점수를 함께 보고, `hybrid_chain`의 답변을 확인합니다.
3. 답변이 비슷한 영화를 정답처럼 소개한다면, 어떻게 고칠 수 있을지 시도합니다.

In [ ]:
# 과제 3: 여기에 작성합니다
